In [1]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException
import pandas as pd
import datetime
from datetime import date
import random
import subprocess
import time as t
import csv
import re
import os

link = "https://www.southernrailway.com"

leaving_from = "Clapham Junction"
going_to = "Brighton"

In [2]:



driver = webdriver.Edge()
driver.get(link)
t.sleep(5)

button = driver.find_element(By.ID, "CybotCookiebotDialogBodyLevelButtonLevelOptinAllowAll")
button.click()
t.sleep(3)

shadow_host = driver.find_element(By.ID, "otrl-custom-hero")
shadow_root = shadow_host.shadow_root
leaving =   shadow_root.find_element(By.NAME, "stationFrom")
leaving.send_keys(leaving_from)
WebDriverWait(driver, 2).until(
    lambda d: len(shadow_root.find_elements(
        By.CSS_SELECTOR, ".otrl-jp__station-autosuggest__item"
    )) > 0
)
first_suggestion = shadow_root.find_element(
    By.CSS_SELECTOR, ".otrl-jp__station-autosuggest__item"
)
first_suggestion.click()

t.sleep(3)
going = shadow_root.find_element(By.NAME, "stationTo")
going.send_keys(going_to)
WebDriverWait(driver, 2).until(
    lambda d: len(shadow_root.find_elements(
        By.CSS_SELECTOR, ".otrl-jp__station-autosuggest__item"
    )) > 0
)
first_suggestion = shadow_root.find_element(
    By.CSS_SELECTOR, ".otrl-jp__station-autosuggest__item"
)
first_suggestion.click()


t.sleep(3)

search = shadow_root.find_element(By.CLASS_NAME, "otrl-jp__tickets__submit")
search.click()
t.sleep(5)
wait = WebDriverWait(driver, 2)
seen_departures = set()  # Track already-captured departure times
records = []

try:
    date_text = driver.find_element(By.CSS_SELECTOR, ".service-carousel-header-v2__date").text.strip()  # e.g. "Tue 03 Mar 2026"
    current_date = datetime.datetime.strptime(date_text, "%a %d %b %Y").date()
except Exception as e:
    print(f"  Could not parse date: {e}")
        
    
current_date_tracker = current_date


def parse_current_page(current_date_tracker):
    """Parse all visible trains and fares, skip already-seen departures."""
       
    
        
    train_items = driver.find_elements(By.CSS_SELECTOR, ".service-box-v2__item")
    fare_tiles  = driver.find_elements(By.CSS_SELECTOR, ".fare-list-v2__tile .price")

    for i, item in enumerate(train_items):
        try:
            sr_text = item.find_element(By.CSS_SELECTOR, ".sr-only").text

            dep_match = re.search(r'^(\d{2}:\d{2})', sr_text)
            arr_match = re.search(r'arriving at .+? at (\d{2}:\d{2})', sr_text)
            dur_match = re.search(r'takes (.+?),', sr_text)
            chg_match = re.search(r'has (\d+) change', sr_text)

            departure = dep_match.group(1) if dep_match else None
            
            if not departure:
                continue

                
                

            arrival  = arr_match.group(1) if arr_match else None
            dep_time = datetime.datetime.strptime(departure, "%H:%M").time()
            current_date = current_date_tracker
            
            if records:
                last_dep_dt = records[-1]["departure_dt"]
                candidate_dt = datetime.datetime.combine(current_date, dep_time)
                # If candidate is more than 6hrs before last departure, bump the date
                if (last_dep_dt - candidate_dt).total_seconds() > 6 * 3600:
                    current_date = current_date + datetime.timedelta(days=1)
                    current_date_tracker = current_date
                    print(f"  Date rolled over to {current_date}")

            dedup_key = (current_date, departure)
            if dedup_key in seen_departures:
                continue

            duration = dur_match.group(1).strip() if dur_match else None
            changes  = int(chg_match.group(1)) if chg_match else 0
            price = None
            if i < len(fare_tiles):
                price_text  = fare_tiles[i].text.strip()
                price_match = re.search(r'[\d.]+', price_text)
                price = float(price_match.group()) if price_match else None

            departure_dt = datetime.datetime.combine(current_date, dep_time)
            arrival_dt   = None
            if arrival:
                arr_time = datetime.datetime.strptime(arrival, "%H:%M").time()
                arrival_dt = datetime.datetime.combine(current_date, arr_time)
                if arrival_dt < departure_dt:
                    arrival_dt += datetime.timedelta(days=1)

            records.append({
                "departure_dt": departure_dt,
                "arrival_dt":   arrival_dt,
                "departure":    departure,
                "arrival":      arrival,
                "duration":     duration,
                "changes":      changes,
                "price_gbp":    price,
            })
            seen_departures.add(dedup_key)
        except Exception as e:
            print(f"  Error parsing train {i}: {e}")

parse_current_page(current_date_tracker)
print(f"After initial load: {len(records)} trains")

for click_num in range(15):
    try:
        t.sleep(random.uniform(0.3, 1.4))
        last_departure_before = driver.find_elements(By.CSS_SELECTOR, ".service-box-v2__item .departure-time time")[-1].get_attribute("datetime")

        later_btn = wait.until(EC.element_to_be_clickable((By.CSS_SELECTOR, "a.service-pager.pull-right")))
        driver.execute_script("arguments[0].click();", later_btn)
        WebDriverWait(driver, 10).until(lambda d: d.find_elements(By.CSS_SELECTOR, ".service-box-v2__item .departure-time time")[-1].get_attribute("datetime") != last_departure_before)
    except TimeoutException:
        print(f"Click {click_num + 1}: page didn't update or no 'Later' button.")
        break
    parse_current_page(current_date_tracker)
    print(f"After click {click_num + 1}: {len(records)} trains total")


driver.quit()
df = pd.DataFrame(records)
print(df.head())

After initial load: 3 trains
After click 1: 5 trains total
After click 2: 7 trains total
After click 3: 9 trains total
After click 4: 11 trains total
After click 5: 13 trains total
After click 6: 15 trains total
After click 7: 17 trains total
After click 8: 18 trains total
After click 9: 19 trains total
After click 10: 20 trains total
After click 11: 22 trains total
  Date rolled over to 2026-03-04
After click 12: 24 trains total
  Date rolled over to 2026-03-04
After click 13: 25 trains total
  Date rolled over to 2026-03-04
After click 14: 27 trains total
  Date rolled over to 2026-03-04
After click 15: 28 trains total
         departure_dt          arrival_dt departure arrival          duration  \
0 2026-03-03 18:01:00 2026-03-03 19:04:00     18:01   19:04  1 hour 3 minutes   
1 2026-03-03 18:12:00 2026-03-03 19:21:00     18:12   19:21  1 hour 9 minutes   
2 2026-03-03 18:31:00 2026-03-03 19:34:00     18:31   19:34  1 hour 3 minutes   
3 2026-03-03 18:42:00 2026-03-03 19:51:00     1